# Israeli 26th Knesset results map

This is the 2026 version of the repository's interactive election map. It replaces the address-based Voronoi regions with real locality boundaries joined to locality totals aggregated from the Central Elections Committee ballot-box export. Hover for a quick summary; click a locality for its party chart.

### Map data

The boundary layer follows the pipeline used by [RavBogard/israelielection](https://github.com/RavBogard/israelielection/blob/main/scripts/vote-map/build.mts): Ministry of Transport 2026 locality polygons, supplemented by dissolved CBS 2008 statistical areas for West Bank localities. Full source URLs and caveats are in [2026/MAP_SOURCES.md](2026/MAP_SOURCES.md).

In [1]:
%matplotlib widget
import random

import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.ticker import PercentFormatter
from ipyleaflet import (
    FullScreenControl, GeoJSON, Map, ScaleControl,
    basemaps,
)
from ipywidgets import HBox, HTML, Label, Layout, VBox

from behirot_2026 import download_election_file, topojson_to_geojson

## Refresh the official results

This downloads and saves the official ballot-box export under 2026/. Locality totals are aggregated from expb.csv, as in the old notebook.

In [2]:
ballot_results = download_election_file("expb.csv")

def clean_export(frame):
    frame = frame.loc[:, ~frame.columns.str.startswith("Unnamed:")].copy()
    numeric_columns = frame.columns.difference(["שם ישוב"])
    frame[numeric_columns] = frame[numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    ).fillna(0)
    return frame

ballot_results = clean_export(ballot_results)

party_column_index = ballot_results.columns.get_loc("קלפי אחרת") + 1
party_columns = list(ballot_results.columns[party_column_index:])
sum_columns = ["מצביעים", "פסולים", "כשרים", *party_columns]
locality_results = (
    ballot_results.groupby("סמל ישוב", as_index=False)
    .agg({"שם ישוב": "first", **{column: "sum" for column in sum_columns}})
)

## Join results to locality boundaries

In [3]:
locality_boundaries = topojson_to_geojson("2026/boundaries.topo.json")
locality_lookup = locality_results.set_index("סמל ישוב")
for feature in locality_boundaries["features"]:
    properties = feature["properties"]
    code = int(properties["code"])
    properties["has_results"] = code in locality_lookup.index
    if not properties["has_results"]:
        continue

    row = locality_lookup.loc[code]
    votes = row[party_columns].astype(int)
    valid_votes = int(row["כשרים"])
    properties.update({
        "name": row["שם ישוב"],
        "voted": int(row["מצביעים"]),
        "invalid": int(row["פסולים"]),
        "valid": valid_votes,
        "votes": {party: int(votes[party]) for party in party_columns},
    })



## Interactive map

In [4]:
plt.ioff()
fig, party_ax = plt.subplots(1, 1)
plt.ion()

def plot_party_votes_df(party_votes, ax):
    ax.cla()
    plot_data = party_votes.sort_values().copy()
    plot_data = plot_data * 100 / plot_data.sum()
    plot_data.index = map(lambda s: s, plot_data.index)
    plot_data.plot(kind='barh', ax=ax)
    ax.set_ylim([len(plot_data)-10, len(plot_data)])
    ax.set_xlim([0,100])
    ax.xaxis.set_major_formatter(PercentFormatter())

def plot_locality_summary(feature, party_ax):
    properties = feature["properties"]
    if not properties.get("has_results"):
        return
    party_votes = pd.Series(properties["votes"])
    plot_party_votes_df(party_votes, party_ax)
    party_ax.figure.suptitle(properties["name"])

title_widget = HTML()

def set_format(fmt):
    def set_in_fmt(widget, variable):
        widget.value = fmt % variable
    return set_in_fmt

set_title = set_format("<h1>%s</h1>")
set_voted = set_format('מצביעים: %s')
set_invalid = set_format('פסולים: %s, %.2f%%')
set_valid = set_format('כשרים: %s, %.2f%%')
set_dominant_party = set_format('מפלגה דומיננטית: %s')
set_dominant_party_votes = set_format('הצבעות למלגה הדומיננטית: %s, %.2f%%')

voted_label = Label(style={"font_size": "23px"})
invalid_label = Label(style={"font_size": "23px"})
valid_label = Label(style={"font_size": "23px"})
dominant_party_label = Label(style={"font_size": "23px", "font_weight": "bold"})
dominant_party_votes_label = Label(style={"font_size": "23px"})

set_voted(voted_label, '')
set_invalid(invalid_label, ('', 0))
set_valid(valid_label, ('', 0))
set_title(title_widget, "Hover to display quick summary")
set_dominant_party(dominant_party_label, '')
set_dominant_party_votes(dominant_party_votes_label, ('', 0))

def get_dominant_party(party):
    total = party.sum()
    if not total:
        return '', 0, 0
    name = party.idxmax()
    votes = party.loc[name]
    return name, votes, votes/total

def update_summary_labels(feature):
    properties = feature["properties"]
    if not properties.get("has_results"):
        set_title(title_widget, "אין עדיין תוצאות ליישוב זה")
        return
    party = pd.Series(properties["votes"])
    set_title(title_widget, properties["name"])
    voted = properties["voted"]
    invalid = properties["invalid"]
    valid = properties["valid"]
    set_voted(voted_label, voted)
    set_valid(valid_label, (valid, valid*100/voted if voted else 0))
    set_invalid(invalid_label, (invalid, invalid*100/voted if voted else 0))
    name, votes, percent = get_dominant_party(party)
    set_dominant_party(dominant_party_label, name)
    set_dominant_party_votes(dominant_party_votes_label, (votes, percent*100))

labels = VBox([title_widget, dominant_party_label, voted_label, valid_label, invalid_label, dominant_party_votes_label])

In [9]:
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(31.5, 34.9),
    zoom=7,
    min_zoom=6,
    max_zoom=15,
    scroll_wheel_zoom=True,
    prefer_canvas=True,
    world_copy_jump=False,
    layout=Layout(height="760px", width="60%"),
)

def random_color(feature):
    return {
        'color': 'black',
        'fillColor': random.choice(['red', 'yellow', 'green', 'orange']),
    }

geo_json = GeoJSON(
    data=locality_boundaries,
    style={
        'opacity': 1, 'dashArray': '9', 'fillOpacity': 0.1, 'weight': 1
    },
    hover_style={
        'color': 'white', 'dashArray': '0', 'fillOpacity': 0.5
    },
    style_callback=random_color,
)
geo_json.on_hover(lambda feature, **kwargs: update_summary_labels(feature))
geo_json.on_click(lambda feature, **kwargs: plot_locality_summary(feature, party_ax=party_ax))
m.add(geo_json)

panel = VBox(
    [labels, fig.canvas],
    layout=Layout(width="40%", max_height="760px"),
)
m.add(ScaleControl(position="bottomleft", imperial=False))
m.add(FullScreenControl(position="topleft"))
map_with_panel = HBox(
    [m, panel],
    layout=Layout(width="100vw"),
);



In [10]:
map_with_panel